# Phase 1 — Data Exploration
Quick look at the generated dataset to confirm it looks right before we build the pipeline.

Run `python src/generate_data.py` first to create `data/raw/rides.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df = pd.read_csv('../data/raw/rides.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Column dtypes and null counts
info = pd.DataFrame({
    'dtype': df.dtypes,
    'nulls': df.isnull().sum(),
    'null_%': (df.isnull().mean() * 100).round(2),
    'unique': df.nunique()
})
info

In [ ]:
# Key descriptive statistics
df[['wait_time_min','surge_multiplier','distance_km','fare_inr','driver_rating']].describe().round(2)

In [ ]:
# ── Plot 1: Hourly ride volume (demand curve) ─────────────────────────────
df['hour'] = df['timestamp'].dt.hour
hourly = df.groupby('hour').size()

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#185FA5' if h in [7,8,9,17,18,19] else '#85B7EB' for h in hourly.index]
ax.bar(hourly.index, hourly.values / 1000, color=colors, width=0.8)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Rides (thousands)')
ax.set_title('Hourly ride volume — all cities combined')
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.savefig('../reports/figures/01_hourly_demand.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: City-wise ride volume ─────────────────────────────────────────
city_counts = df['city'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(city_counts.index[::-1], city_counts.values[::-1] / 1000, color='#185FA5')
ax.set_xlabel('Rides (thousands)')
ax.set_title('Total rides by city')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.savefig('../reports/figures/02_city_volume.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Surge distribution ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
df['surge_multiplier'].hist(bins=30, color='#185FA5', alpha=0.8, ax=ax)
ax.set_xlabel('Surge multiplier')
ax.set_ylabel('Ride count')
ax.set_title('Surge multiplier distribution')
plt.tight_layout()
plt.savefig('../reports/figures/03_surge_distribution.png', bbox_inches='tight')
plt.show()

print('Surge breakdown:')
bins = [1.0, 1.01, 1.5, 2.0, 2.5, 4.1]
labels = ['No surge (1×)', 'Low (1–1.5×)', 'Med (1.5–2×)', 'High (2–2.5×)', 'Extreme (>2.5×)']
for label, lo, hi in zip(labels, bins, bins[1:]):
    pct = ((df['surge_multiplier'] >= lo) & (df['surge_multiplier'] < hi)).mean() * 100
    print(f'  {label:<22} {pct:.1f}%')

In [ ]:
# ── Plot 4: Festival vs normal day ride volume ────────────────────────────
fest = df.groupby('is_festival').agg(
    avg_wait=('wait_time_min', 'mean'),
    avg_surge=('surge_multiplier', 'mean'),
    avg_fare=('fare_inr', 'mean'),
    ride_count=('ride_id', 'count')
).round(2)
fest.index = ['Normal day', 'Festival day']
print('Festival day effect:')
print(fest.to_string())

In [ ]:
# ── Plot 5: Weather impact on wait times ─────────────────────────────────
weather_wait = df.groupby('weather')['wait_time_min'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors_w = {'heavy_rain':'#E24B4A', 'fog':'#BA7517', 'light_rain':'#378ADD', 'clear':'#1D9E75'}
bars = ax.bar(weather_wait.index, weather_wait.values,
              color=[colors_w.get(w, '#888') for w in weather_wait.index])
ax.set_ylabel('Median wait time (min)')
ax.set_title('Weather impact on wait times')
for bar, val in zip(bars, weather_wait.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f} min', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/04_weather_wait.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Phase 1 complete!')
print('All figures saved to reports/figures/')
print('Next: run python src/pipeline.py  (Phase 2)')